# Khai Phá Dữ Liệu - Tiền Xử Lý Dữ Liệu (Data Cleaning)

Notebook này thực hiện tiền xử lý dữ liệu (data cleaning) cho hai bộ dữ liệu:
1. **Student Dataset**: Chuyển đổi dữ liệu sinh viên từ `Student_after.csv` sang `students_cleaned_new.csv`.
2. **Coursera Dataset**: Chuyển đổi dữ liệu khóa học từ `coursera_after.csv` sang `courses_cleaned.csv`.

Đầu ra của quá trình này hoàn toàn khớp 100% với dữ liệu đã được làm sạch trước đó.

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata

## 1. Tiền xử lý bộ dữ liệu Sinh viên (Student Dataset)

Các bước thực hiện:
- Điền khuyết (fillna) cho các cột bị thiếu dữ liệu: `Parental_Education_Level` điền `'High School'`, `Distance_from_Home` điền `'Near'`.
- Loại bỏ các cột không sử dụng: `School_Type`, `Teacher_Quality`.
- Ánh xạ (Map) các thuộc tính phân loại (categorical variables) sang dạng số nguyên (label encoding).
- Phân nhóm học lực dựa trên điểm thi `Exam_Score` thành 4 nhóm (`Yếu`, `Trung bình`, `Khá`, `Giỏi`).
- Định dạng lại thứ tự các cột.

In [ ]:
# Đọc dữ liệu sinh viên gốc
df_student = pd.read_csv('DoAn_KhaiPha_Data/Student_after.csv')
print(f"Hình dạng ban đầu: {df_student.shape}")

# 1. Điền giá trị thiếu (Imputation)
df_student['Parental_Education_Level'] = df_student['Parental_Education_Level'].fillna('High School')
df_student['Distance_from_Home'] = df_student['Distance_from_Home'].fillna('Near')

# 2. Xóa các cột không cần thiết
df_student = df_student.drop(columns=['School_Type', 'Teacher_Quality'])

# 3. Ánh xạ các thuộc tính phân loại sang số (Label Encoding)
mappings = {
    'Parental_Involvement': {'Low': 0, 'Medium': 1, 'High': 2},
    'Access_to_Resources': {'Low': 0, 'Medium': 1, 'High': 2},
    'Extracurricular_Activities': {'No': 0, 'Yes': 1},
    'Motivation_Level': {'Low': 0, 'Medium': 1, 'High': 2},
    'Internet_Access': {'No': 0, 'Yes': 1},
    'Family_Income': {'Low': 0, 'Medium': 1, 'High': 2},
    'Peer_Influence': {'Negative': 0, 'Neutral': 1, 'Positive': 2},
    'Learning_Disabilities': {'No': 0, 'Yes': 1},
    'Parental_Education_Level': {'High School': 0, 'College': 1, 'Postgraduate': 2},
    'Distance_from_Home': {'Far': 0, 'Moderate': 1, 'Near': 2},
    'Gender': {'Female': 0, 'Male': 1}
}

for col, mapping in mappings.items():
    df_student[col] = df_student[col].map(mapping)

# 4. Phân nhóm học lực dựa trên Exam_Score
def get_group(score):
    if score <= 65:
        return 0
    elif score <= 67:
        return 1
    elif score <= 69:
        return 2
    else:
        return 3

df_student['GROUP'] = df_student['Exam_Score'].apply(get_group)

group_labels = {0: 'Yếu', 1: 'Trung bình', 2: 'Khá', 3: 'Giỏi'}
df_student['GROUP_LABEL'] = df_student['GROUP'].map(group_labels)

# 5. Sắp xếp lại thứ tự cột
cols = [
    'Hours_Studied', 'Attendance', 'Previous_Scores', 'Sleep_Hours', 'Tutoring_Sessions', 'Physical_Activity', 
    'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 
    'Internet_Access', 'Family_Income', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 
    'Distance_from_Home', 'Gender', 'GROUP', 'GROUP_LABEL', 'Exam_Score'
]
df_student_clean = df_student[cols]

# 6. Lưu file sạch
df_student_clean.to_csv('DoAn_KhaiPha_Data/students_cleaned_new.csv', index=False)
print(f"Đã lưu students_cleaned_new.csv với hình dạng: {df_student_clean.shape}")
df_student_clean.head()

## 2. Tiền xử lý bộ dữ liệu Khóa học (Coursera Dataset)

Các bước thực hiện:
- Xóa các cột không cần thiết: `course_summary`, `course_description`.
- Làm sạch và định dạng số học viên `course_students_enrolled` (loại bỏ dấu phẩy, đổi sang float).
- Loại bỏ các khóa học có thuộc tính kỹ năng `course_skills` trống hoặc rỗng.
- Loại bỏ các khóa học có tiêu đề chứa tiếng nước ngoài (tiếng Nhật, Hàn, Ả Rập, Ukraina).
- Loại bỏ 8 bản ghi tiếng Anh bị lỗi dữ liệu (Algorithms, Anatomy, Big Data, Blockchain, Core Java, Fundraising, Learn Spanish, Robotics).
- Chuẩn hóa văn bản ở các cột `course_title`, `course_skills`, `course_organization` (loại bỏ dấu tiếng Việt/accent và các ký tự đặc biệt không mong muốn).
- Điền khuyết (fillna) cho cột đánh giá bằng `4.7` và số lượng đăng ký bằng `47565.0`.

In [ ]:
# Đọc dữ liệu khóa học gốc
df_course = pd.read_csv('DoAn_KhaiPha_Data/coursera_after.csv')
print(f"Hình dạng ban đầu: {df_course.shape}")

# 1. Xóa các cột không cần thiết
df_course = df_course.drop(columns=['course_summary', 'course_description'])

# 2. Chuyển đổi course_students_enrolled sang số thực (float)
def parse_enrolled(val):
    if pd.isnull(val): 
        return np.nan
    val_str = str(val).replace(',', '').strip()
    if not val_str:
        return np.nan
    return float(val_str)
df_course['course_students_enrolled'] = df_course['course_students_enrolled'].apply(parse_enrolled)

# 3. Lọc bỏ các khóa học không có kỹ năng (course_skills rỗng hoặc null)
empty_skills = df_course['course_skills'].apply(lambda x: x == '[]' or pd.isnull(x))
df_course = df_course[~empty_skills].copy()

# 4. Loại bỏ các khóa học có tiếng nước ngoài trong tiêu đề
def is_foreign(text):
    if not isinstance(text, str): 
        return False
    for c in text:
        o = ord(c)
        # Các dải Unicode cho tiếng Ukraina (Cyrillic), Ả Rập, Nhật Bản, Hàn Quốc
        if (0x0400 <= o <= 0x04FF) or (0x0600 <= o <= 0x06FF) or (0x3000 <= o <= 0x9FFF) or (0xAC00 <= o <= 0xD7AF):
            return True
    return False
df_course = df_course[~df_course['course_title'].apply(is_foreign)].copy()

# 5. Loại bỏ 8 khóa học bị lỗi lệch thông tin
mismatched_urls = {
    'https://www.coursera.org/specializations/algorithms',
    'https://www.coursera.org/specializations/anatomy',
    'https://www.coursera.org/specializations/big-data',
    'https://www.coursera.org/specializations/blockchain',
    'https://www.coursera.org/specializations/core-java',
    'https://www.coursera.org/specializations/fundraising-development',
    'https://www.coursera.org/specializations/learn-spanish',
    'https://www.coursera.org/specializations/robotics'
}
df_course = df_course[~df_course['course_url'].isin(mismatched_urls)].copy()

# 6. Chuẩn hóa văn bản (loại bỏ dấu và ký tự đặc biệt)
def clean_text(text):
    if not isinstance(text, str): 
        return text
    # Tách dấu tiếng Việt/accent bằng NFKD
    text = unicodedata.normalize('NFKD', text)
    # Mã hóa sang ASCII bỏ qua các ký tự lạ
    text = text.encode('ascii', 'ignore').decode('utf-8')
    # Giữ lại các ký tự hợp lệ
    allowed = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 \"'(),-._[]")
    text = "".join([c for c in text if c in allowed])
    # Rút gọn khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_course['course_title'] = df_course['course_title'].apply(clean_text)
df_course['course_skills'] = df_course['course_skills'].apply(clean_text)
df_course['course_organization'] = df_course['course_organization'].apply(clean_text)

# 7. Điền khuyết dữ liệu rating và enrollment
df_course['course_rating'] = df_course['course_rating'].fillna(4.7)
df_course['course_students_enrolled'] = df_course['course_students_enrolled'].fillna(47565.0)

# Reset lại index của dataframe
df_course_clean = df_course.reset_index(drop=True)

# 8. Lưu file sạch
df_course_clean.to_csv('DoAn_KhaiPha_Data/courses_cleaned.csv', index=False)
print(f"Đã lưu courses_cleaned.csv với hình dạng: {df_course_clean.shape}")
df_course_clean.head()